# B&W Photography Clustering & Aesthetic Analysis Pipeline

This notebook implements a sequential, memory-safe pipeline for processing black and white photography.
It runs on Apple Silicon (MPS) and uses Qwen3-VL models for both embedding extraction and captioning.

**Hardware Profile:** Apple M4 Pro (24GB Unified Memory)  
**Models:** 
- `Qwen/Qwen3-VL-Embedding-2B` (Stage 1: Embeddings)
- `Qwen/Qwen3-VL-8B-Instruct` (Stage 2: Captions)

In [ ]:
import os
import gc
import json
import warnings
import hashlib
from pathlib import Path
from collections import defaultdict

import torch
import duckdb
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import umap
from sklearn.cluster import HDBSCAN
from transformers import AutoProcessor, AutoModel, AutoModelForCausalLM

# Optimize MPS memory allocation
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"
warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"[✓] Using device: {DEVICE}")

DATA_DIR = Path("./data/raw")
DB_PATH = Path("./outputs/photos.duckdb")
RESULTS_PATH = Path("./outputs/results.json")
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
def get_file_hash(path: Path) -> str:
    """Generate a short stable hash for a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(65536))
    return h.hexdigest()[:16]

def safe_to_rgb(image_path: Path) -> Image.Image:
    """Robustly convert any image to 3-channel RGB."""
    img = Image.open(image_path)
    
    if img.mode in ("RGBA", "LA", "PA"):
        # Composite over white background to avoid blacking out transparent regions
        background = Image.new("RGB", img.size, (255, 255, 255))
        if img.mode == "PA":
            img = img.convert("RGBA")
        background.paste(img, mask=img.split()[-1])
        return background
    elif img.mode in ("L", "1", "P", "CMYK", "YCbCr"):
        return img.convert("RGB")
    return img.convert("RGB")

def discover_images(directory: Path):
    valid_exts = {".jpg", ".jpeg", ".png", ".tiff", ".webp"}
    images = []
    for f in directory.rglob("*"):
        if f.suffix.lower() in valid_exts and f.is_file():
            images.append(f)
    return sorted(images)

images_paths = discover_images(DATA_DIR)
print(f"[✓] Found {len(images_paths)} valid images.")

In [ ]:
print("[i] Stage 1: Loading Qwen3-VL-Embedding-2B...")
embed_model_id = "Qwen/Qwen3-VL-Embedding-2B"

con = duckdb.connect(str(DB_PATH))
con.execute("""
    CREATE TABLE IF NOT EXISTS style_qwen3 (
        photo_id   VARCHAR PRIMARY KEY,
        image_path VARCHAR NOT NULL,
        embedding  FLOAT[] NOT NULL,
        created_at TIMESTAMP DEFAULT now()
    )
""")

processor = AutoProcessor.from_pretrained(embed_model_id)
embed_model = AutoModel.from_pretrained(
    embed_model_id,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map="mps"
)
embed_model.eval()

embeddings = []
valid_paths = []

print("[i] Processing images...")
for path in tqdm(images_paths, desc="Stage 1"):
    pid = get_file_hash(path)
    
    # Check if already in DB
    existing = con.execute("SELECT embedding FROM style_qwen3 WHERE photo_id = ?", [pid]).fetchone()
    if existing:
        emb = np.array(existing[0])
        embeddings.append(emb)
        valid_paths.append(path)
        continue
        
    try:
        img_rgb = safe_to_rgb(path)
        messages = [
            {"role": "user", "content": [{"type": "image", "image": img_rgb}, {"type": "text", "text": ""}]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img_rgb], return_tensors="pt", padding=True).to(DEVICE)
        
        with torch.inference_mode():
            outputs = embed_model(**inputs)
            last_hidden = outputs.last_hidden_state[:, -1, :]
            
        emb = last_hidden.detach().cpu().to(torch.float32).numpy().flatten()
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        
        # Save to DB
        con.execute("""
            INSERT INTO style_qwen3 (photo_id, image_path, embedding)
            VALUES (?, ?, ?)
        """, [pid, str(path.resolve()), emb.tolist()])
        
        embeddings.append(emb)
        valid_paths.append(path)
    except Exception as e:
        print(f"[!] Error processing {path.name}: {e}")

con.close()
embeddings = np.array(embeddings)
print(f"[✓] Loaded {len(embeddings)} embeddings.")

In [ ]:
print("[i] Unloading Embedding Model & Clearing Memory...")
del embed_model
del processor
torch.mps.empty_cache()
gc.collect()
print("[✓] Memory cleared")

In [ ]:
print("[i] Clustering: UMAP -> HDBSCAN...")
if len(embeddings) > 15: # Safety check
    reducer = umap.UMAP(n_components=50, metric='cosine', random_state=42)
    reduced_embs = reducer.fit_transform(embeddings)

    clusterer = HDBSCAN(min_cluster_size=5, metric='euclidean')
    labels = clusterer.fit_predict(reduced_embs)
else:
    print("[!] Not enough samples for meaningful clustering. Assigning all to cluster 0.")
    labels = np.zeros(len(embeddings), dtype=int)

clusters = defaultdict(list)
for path, label in zip(valid_paths, labels):
    clusters[int(label)].append(path)

print(f"[✓] Found {len(set(labels)) - (1 if -1 in labels else 0)} clusters (excluding noise).")

In [ ]:
print("[i] Stage 2: Loading Qwen3-VL-8B-Instruct...")
instruct_model_id = "Qwen/Qwen3-VL-8B-Instruct"

processor = AutoProcessor.from_pretrained(instruct_model_id)
instruct_model = AutoModelForCausalLM.from_pretrained(
    instruct_model_id,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map="mps"
)
instruct_model.eval()

cluster_results = []
prompt = "Describe this black and white photograph as an art critic focusing on composition, tonal contrast, and emotional mood."

print("[i] Generating captions for cluster representatives...")
for cluster_id, paths in tqdm(clusters.items(), desc="Captioning"):
    rep_path = paths[0] # Pick the first image as representative
    try:
        img_rgb = safe_to_rgb(rep_path)
        messages = [
            {"role": "user", "content": [{"type": "image", "image": img_rgb}, {"type": "text", "text": prompt}]}
        ]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img_rgb], return_tensors="pt", padding=True).to(DEVICE)
        
        with torch.inference_mode():
            generated_ids = instruct_model.generate(
                **inputs, 
                max_new_tokens=150, 
                temperature=0.7, 
                top_p=0.9,
                do_sample=True
            )
        
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        caption = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
        
        cluster_results.append({
            "cluster_id": cluster_id,
            "representative_path": str(rep_path),
            "caption": caption.strip(),
            "member_count": len(paths),
            "sample_images": [str(p) for p in paths]
        })
    except Exception as e:
        print(f"[!] Error captioning {rep_path.name}: {e}")

In [ ]:
print("[i] Unloading Instruct Model & Clearing Memory...")
del instruct_model
del processor
torch.mps.empty_cache()
gc.collect()
print("[✓] Memory cleared")

In [ ]:
print("[i] Generating Visualizations...")

# --- 1. UMAP Manifold Visualization ---
if 'reduced_embs' in locals() and len(reduced_embs) > 0:
    # We use 2D for the plot, so we reduce to 2 if we haven't already
    if reduced_embs.shape[1] > 2:
        plot_reducer = umap.UMAP(n_components=2, random_state=42)
        vis_data = plot_reducer.fit_transform(embeddings)
    else:
        vis_data = reduced_embs

    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(vis_data[:, 0], vis_data[:, 1], c=labels, cmap='Spectral', s=50, alpha=0.6, edgecolors='w')
    plt.colorbar(scatter, label='Cluster ID')
    plt.title("UMAP Projection of Photo Library (Artistic Metric Space)", fontsize=15)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

# --- 2. Enhanced Cluster Gallery ---
from IPython.display import display, HTML

for res in sorted(cluster_results, key=lambda x: x["cluster_id"]):
    cluster_id = res["cluster_id"]
    name = "Noise (Outliers)" if cluster_id == -1 else f"Cluster {cluster_id}"
    member_paths = res["sample_images"]
    
    print(f"\n{'='*80}")
    print(f"  {name.upper()} | {res['member_count']} Images")
    print(f"{'='*80}")
    
    # Create a layout with the representative image on the left and description on the right
    fig = plt.figure(figsize=(15, 6))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.5])
    
    # Main Image
    ax_main = fig.add_subplot(gs[0, 0])
    main_img = Image.open(res["representative_path"]).convert("RGB")
    ax_main.imshow(main_img)
    ax_main.set_title("Cluster Representative", fontsize=12, fontweight='bold')
    ax_main.axis("off")
    
    # Description Text
    ax_text = fig.add_subplot(gs[0, 1])
    ax_text.axis("off")
    ax_text.text(0.0, 0.95, f"Art Critic Analysis:", fontsize=14, fontweight='bold', verticalalignment='top')
    
    # Wrap text manually for the plot
    words = res["caption"].split()
    wrapped_caption = ""
    line = ""
    for word in words:
        if len(line + word) > 60:
            wrapped_caption += line + "\n"
            line = word + " "
        else:
            line += word + " "
    wrapped_caption += line
    
    ax_text.text(0.0, 0.85, wrapped_caption, fontsize=12, verticalalignment='top', style='italic')
    
    plt.tight_layout()
    plt.show()
    
    # Row of small sample thumbnails from the same cluster
    if len(member_paths) > 1:
        samples_to_show = min(6, len(member_paths))
        fig, axes = plt.subplots(1, samples_to_show, figsize=(15, 3))
        if samples_to_show == 1: axes = [axes]
        
        for i in range(samples_to_show):
            s_img = Image.open(member_paths[i]).convert("RGB")
            axes[i].imshow(s_img)
            axes[i].axis("off")
        
        plt.suptitle(f"Sample members from {name}", fontsize=10, y=1.05)
        plt.show()


In [ ]:
print("[i] Exporting Results...")
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(cluster_results, f, indent=2)
print(f"[✓] Results saved to {RESULTS_PATH}")